In [ ]:
from google.colab import drive
import os
import pandas as pd
from rpy2.robjects import r, pandas2ri
from datetime import datetime

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
AU_DIR = '/content/drive/MyDrive/au_activity'

REAL = f'{AU_DIR}/real'
PSEUDO = f'{AU_DIR}/pseudopairs'
WLCC = f'{AU_DIR}/wlcc'

DATA_DIR = '/content/drive/MyDrive/pioneer_data'

pseudo_metadata = pd.read_csv(f'{DATA_DIR}/pseudo_metadata.csv')
asym_data = pd.read_csv(f'{DATA_DIR}/asym_data.csv')

In [ ]:
pseudo_metadata.head()

,pair_id,dir_num,convo_id,user_id,channel,enjoyment,enjoy_diff
0,0,36,b88c01b7-4d38-42c5-b650-ee796b3ef869,5dd33bd8f01cbe3350807cf6,L,4.0,0.0
1,0,12,2fb5e5af-52fe-4469-b77a-0b4467ca0bf1,5d54d3560c5b0400174b3c81,R,4.0,0.0
2,1,12,2fb5e5af-52fe-4469-b77a-0b4467ca0bf1,5e2be6efe179572183644844,L,8.0,0.0
3,1,49,671f9e6e-a35f-4bdc-9b61-4adb53bb1690,5e531b2205acdb33c0f5f24c,R,8.0,0.0
4,2,106,f00d9688-27fe-45ce-9d26-e38f729e8641,5d0fb996286e1700010de2c1,L,4.0,-1.0


In [ ]:
asym_data.head()

,dir_num,convo_id,left_id,left_enjoy,right_id,right_enjoy,enjoy_diff
0,3,46f8e9b8-f80a-48cf-90a0-2e29908202c0,5d5eeb06d8bcde00162d73f1,2.0,57656f6c2bfddf000125cce5,6.0,-4.0
1,4,a4fa5355-74ba-4622-a58b-57335efc8a9e,56802e5fc5767f00121cc6a0,9.0,5ecf4b18f7b0443609e07646,5.0,4.0
2,7,5a307dec-0265-4dab-ade6-bc6392695c9e,5e7072e5fb136c63e94f3fa4,5.0,5ca6bbf13b5fcf00100996e9,9.0,-4.0
3,8,7f717277-d9ae-4dbc-b520-0b741d91a6c7,5dae16d241fbb6001160ce72,3.0,5ea9c6541eb4f0121a911e1a,7.0,-4.0
4,10,d9e1f5a5-e4eb-43df-910b-d9ae2befb039,5f3864a4596925371d23631e,5.0,5bf3761862e1bc0001f15cb2,9.0,-4.0


In [ ]:
# testing rMEA on one dyad on only AU6 before scaling

# check for file integrity first
cid = asym_data.iloc[0]['convo_id']

left_df = pd.read_csv(f'{REAL}/{cid}/p_left.csv')
right_df = pd.read_csv(f'{REAL}/{cid}/p_right.csv')

au06 = pd.DataFrame()
au06['left'] = left_df['AU06']
au06['right'] = right_df['AU06']

print(f'shape: {au06.shape}')
print(f'nan count: left={au06['left'].isna().sum()}, right={au06['right'].isna().sum()}')

au06.head()

shape: (10008, 2)
nan count: left=3, right=0


,left,right
0,0.120825,0.000000
1,0.092639,0.000000
2,0.094654,0.000000
3,0.096421,0.000000
4,0.021115,0.001039


In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages('rMEA')
library(rMEA)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/rMEA_1.2.2.tar.gz'
Content type 'application/x-gzip' length 935441 bytes (913 KB)
downloaded 913 KB


The downloaded source packages are in
	‘/tmp/RtmptbMVRE/downloaded_packages’


In [ ]:
%%R -i au06

library(rMEA)

# build one mea object from the two cols of au06 df
mea = MEA(au06, sampRate=6, id='test', session=1, group='real', s1Name='left', s2Name='right')

# preprocess like diao's: 0.5s moving avg, scale by sd
# smoothing is centered since window is symmetric, doesnt shift lag
mea = MEAsmooth(mea, moving.average.win=0.5)
mea = MEAscale(mea, scale='sd', center=FALSE)

cat('number of NAs after processing (left): ', sum(is.na(mea$MEA$left)),
    'right: ', sum(is.na(mea$MEA$right)), '\n')

mea = MEAccf(mea, lagSec=5, winSec=30, incSec=15, r2Z=TRUE, ABS=TRUE)

cat('ccf table: ', nrow(mea$ccf), 'windows x', ncol(mea$ccf), 'lags\n')
cat('NA cells in ccf: ', sum(is.na(mea$ccf)), '\n')

ccfResNames(mea)


Moving average smoothing:

Rescaling data:
number of NAs after processing (left):  3 right:  0 
ccf table:  109 windows x 61 lags
NA cells in ccf:  183 
[1] "all_lags"  "s1_lead"   "s2_lead"   "lag_zero"  "s1_lead_0" "s2_lead_0"
[7] "bestLag"   "grandAver" "winTimes" 


In [ ]:
%%R

avg <- getCCF(mea, type='grandAver')
cat('global average: ', avg, '\n')
# cat(mea$ccfRes$grandAver)

lagzero <- getCCF(mea, type='lag_zero')
cat('lag zero: ', mean(lagzero, na.rm=TRUE), '\n')

global average:  0.1339048 
lag zero:  0.133763 


In [ ]:
%%R
library(rMEA)

# function to compute wlcc and extract metrics
compute_wlcc <- function(dyad) {

    # converts data into motion energy analysis object
    mea <- MEA(dyad, sampRate=6, id='x', session=1, group='x', s1Name='left', s2Name='right')

    # hide warnings
    invisible(capture.output({
        mea <- MEAsmooth(mea, moving.average.win=0.5)
        mea <- MEAscale(mea, scale='sd', center=FALSE)

        # do ccf twice: one with abs=true for strength, and one with abs=false for directionality
        abs_ccf = MEAccf(mea, lagSec=5, winSec=30, incSec=15, r2Z=TRUE, ABS=TRUE)
        sign_ccf = MEAccf(mea, lagSec=5, winSec=30, incSec=15, r2Z=TRUE, ABS=FALSE)
    }))

    # extract metrics

    # number of windows CCF created; more rows equates to higher data stability
    n_windows = nrow(abs_ccf$ccf)

    # global average synchrony across all windows; takes abs value
    grand_aver = abs_ccf$ccfRes$grandAver

    # simultaneous with lag=zero synchrony values
    lag_zero <- abs_ccf$ccfRes$lag_zero

    # peak synchrony in each window
    peak_sync <- suppressWarnings(apply(as.matrix(abs_ccf$ccf), 1, max, na.rm=TRUE))
    peak_sync[is.infinite(peak_sync)] <- NA

    # the lags with highest correlation in each window
    best_lag <- sign_ccf$ccfRes$bestLag

    # coordination when left-leading lag tested
    s1 <- sign_ccf$ccfRes$s1_lead

    # coordination when right-leading lag tested
    s2 <- sign_ccf$ccfRes$s2_lead

    # difference between the two; answers who really led and by how much
    signed_lead <- s1 - s2

    # creates named list to return to python where it becomes a dictionary
    list(
        n_windows = n_windows,

        grand_aver = grand_aver,

        lag_zero_mean = mean(lag_zero, na.rm=TRUE),
        lag_zero_std = sd(lag_zero, na.rm=TRUE),

        peak_sync_mean = mean(peak_sync, na.rm=TRUE),
        peak_sync_std = sd(peak_sync, na.rm=TRUE),

        best_lag_mean = mean(best_lag, na.rm=TRUE),
        best_lag_std = sd(best_lag, na.rm=TRUE),

        signed_lead_mean = mean(signed_lead, na.rm=TRUE),
        signed_lead_std = sd(signed_lead, na.rm=TRUE),

        s1_lead_mean = mean(s1, na.rm=TRUE),
        s1_lead_std = sd(s1, na.rm=TRUE),

        s2_lead_mean = mean(s2, na.rm=TRUE),
        s2_lead_std = sd(s2, na.rm=TRUE)
    )
}

In [ ]:
pandas2ri.activate()
compute_wlcc = r['compute_wlcc']

AUS = ['AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11',
       'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26',
       'AU28', 'AU43']

MIN_LEN = (30 + 5) * 6 # 210 samples needed for one window

In [ ]:
def get_metrics(dyad):
  res = compute_wlcc(dyad)
  return dict(zip(res.names, [v[0] for v in res]))

def run_au(au):
  rows = []

  # iterate through real convos first
  for r in asym_data.itertuples():
    left = pd.read_csv(f'{REAL}/{r.convo_id}/p_left.csv')
    right = pd.read_csv(f'{REAL}/{r.convo_id}/p_right.csv')

    if len(left) < MIN_LEN or len(right) < MIN_LEN:
      print(f'skipping real convo ({r.convo_id}: too short)')
      continue

    try:
      metrics = get_metrics(pd.DataFrame({'left': left[au], 'right': right[au]}))

      rows.append({
          'type': 'real',
          'id': r.convo_id,
          'left_id': r.left_id,
          'right_id': r.right_id,
          'left_enjoy': r.left_enjoy,
          'right_enjoy': r.right_enjoy,
          'enjoy_diff': r.left_enjoy - r.right_enjoy,
          **metrics
      })
    except Exception as e:
      print(f'skipping real convo ({r.convo_id}): {e}')

  # now iterate through pseudopairings
  for pid, group in pseudo_metadata.groupby('pair_id'):
    try:
      left = group[group.channel == 'L'].iloc[0]
      right = group[group.channel == 'R'].iloc[0]

      left_df = pd.read_csv(f'{PSEUDO}/{pid}/p_left.csv')
      right_df = pd.read_csv(f'{PSEUDO}/{pid}/p_right.csv')

      if len(left_df) < MIN_LEN or len(right_df) < MIN_LEN:
        print(f'skipping pseudopair ({pid}: too short)')
        continue

      metrics = get_metrics(pd.DataFrame({'left': left_df[au], 'right': right_df[au]}))
      rows.append({
          'type': 'pseudo',
          'id': pid,
          'left_id': left.user_id,
          'right_id': right.user_id,
          'left_enjoy': left.enjoyment,
          'right_enjoy': right.enjoyment,
          'enjoy_diff': left.enjoyment - right.enjoyment,
          **metrics
      })
    except Exception as e:
      print(f'skipping pseudopair ({pid}): {e}')

  df = pd.DataFrame(rows)
  df.to_csv(f'{WLCC}/{au}.csv', index=False)
  print(f'{au}: {len(df)} rows')
  return df

In [ ]:
total_start = datetime.now()

for au in AUS:
  print(f'starting {au}...')
  au_start = datetime.now()
  run_au(au)
  au_finish = datetime.now()
  print(f'finished {au} in {au_finish - au_start}')

total_finish = datetime.now()

print(f'finished all in {total_finish - total_start}')

print('all done!')

starting AU01...
AU01: 222 rows
finished AU01 in 0:02:05.279463
starting AU02...
AU02: 222 rows
finished AU02 in 0:01:54.973770
starting AU04...
AU04: 222 rows
finished AU04 in 0:01:53.001389
starting AU05...
AU05: 222 rows
finished AU05 in 0:01:56.957363
starting AU06...
AU06: 222 rows
finished AU06 in 0:01:54.539379
starting AU07...
AU07: 222 rows
finished AU07 in 0:01:49.819539
starting AU09...
AU09: 222 rows
finished AU09 in 0:01:59.520208
starting AU10...
AU10: 222 rows
finished AU10 in 0:01:51.304107
starting AU11...
AU11: 222 rows
finished AU11 in 0:02:14.619353
starting AU12...
AU12: 222 rows
finished AU12 in 0:01:53.998033
starting AU14...
AU14: 222 rows
finished AU14 in 0:02:02.879894
starting AU15...
AU15: 222 rows
finished AU15 in 0:01:53.923587
starting AU17...
AU17: 222 rows
finished AU17 in 0:01:54.515560
starting AU20...
AU20: 222 rows
finished AU20 in 0:01:55.495851
starting AU23...
AU23: 222 rows
finished AU23 in 0:01:54.374999
starting AU24...
AU24: 222 rows
finished